In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from shapely.geometry import Point
import random
from datetime import datetime

# Reload what you already have
restaurants_gdf = gpd.read_file('../Cleaned Data/restaurants.geojson')
dispatch_gdf = gpd.read_file('../Cleaned Data/dispatch_centers_WGS84.csv')
G = ox.graph_from_place("Manhattan, New York, USA", network_type='bike')

# Your Midtown dispatch center
origin_lat = 40.764
origin_lon = -73.980
origin_node = ox.nearest_nodes(G, origin_lon, origin_lat)

print(f"Loaded {len(restaurants_gdf)} restaurants")
print(f"Network: {len(G.nodes)} nodes, {len(G.edges)} edges")
print(f"Origin node: {origin_node}")

Loaded 21406 restaurants
Network: 7259 nodes, 15072 edges
Origin node: 4544486782


In [2]:
import openpyxl

# Check what's in your Excel files
perf = pd.read_excel('../Cleaned Data/Mode_performance_parameters.xlsx')
cost = pd.read_excel('../Cleaned Data/Mode_cost_parameters.xlsx')

print("=== Performance Parameters ===")
print(perf.head())
print("\n=== Cost Parameters ===")
print(cost.head())

=== Performance Parameters ===
   S.No       Modes  Max payload(Kg)  Service time(min)/Stop  Stops/ trip  \
0     1       Truck             1200                      30           25   
1     2       E-Van             1000                      30           40   
2     3  Cargo Bike              300                      10           13   

   Parcels/stop  Trips / day  Max Parcel /day  Range(Km)/ day  \
0             5            1              125             250   
1             2            2              160             200   
2             1            3               39              80   

   Min Service Radius  Km/ day  Max Service Radius  Km/ day  \
0                           12                           40   
1                           10                           30   
2                            2                            6   

   Min Daily service Km  Max Daily service Km  Max Allowed Speed(kmh)  \
0                    20                    50                      40   


In [3]:
class ModeParameters:
    """Load and store parameters for each delivery mode"""
    
    def __init__(self, perf_df, cost_df):
        self.params = {}
        
        for idx, row in perf_df.iterrows():
            mode = row['Modes']
            self.params[mode] = {
                # Capacity constraints
                'max_payload_kg': row['Max payload(Kg)'],
                'max_parcels_per_day': row['Max Parcel /day'],
                'stops_per_trip': row['Stops/ trip'],
                'trips_per_day': row['Trips / day'],
                
                # Distance constraints
                'max_service_radius_km': row['Max Service Radius  Km/ day'],
                'max_daily_km': row['Max Daily service Km'],
                
                # Speed and time
                'speed_kmh': row['Max Allowed Speed(kmh)'],
                'service_time_min': row['Service time(min)/Stop'],
                'congestion_sensitivity': row['Congestion Sensitivity'],
                'operating_hours': row['Operating Hours'],
                
                # Environmental
                'emission_factor_co2_per_km': row['Emission Factor (CO2/Km)']
            }
            
            # Add cost parameters
            cost_row = cost_df[cost_df['Modes'] == mode].iloc[0]
            self.params[mode]['cost_per_km'] = cost_row['Cost_per_km']
            self.params[mode]['energy_cost'] = cost_row['Energy/ Fuel Cost']
    
    def get(self, mode):
        return self.params[mode]
    
    def print_summary(self):
        for mode, params in self.params.items():
            print(f"\n{mode}:")
            print(f"  Max parcels/day: {params['max_parcels_per_day']}")
            print(f"  Max service radius: {params['max_service_radius_km']} km")
            print(f"  Speed: {params['speed_kmh']} km/h")
            print(f"  Cost per km: ${params['cost_per_km']}")
            print(f"  Emissions: {params['emission_factor_co2_per_km']} kg CO2/km")

# Initialize
mode_params = ModeParameters(perf, cost)
mode_params.print_summary()


Truck:
  Max parcels/day: 125
  Max service radius: 40 km
  Speed: 40 km/h
  Cost per km: $1.2
  Emissions: 0.3235 kg CO2/km

E-Van:
  Max parcels/day: 160
  Max service radius: 30 km
  Speed: 50 km/h
  Cost per km: $0.7
  Emissions: 0.1575 kg CO2/km

Cargo Bike:
  Max parcels/day: 39
  Max service radius: 6 km
  Speed: 25 km/h
  Cost per km: $0.25
  Emissions: 0.0155 kg CO2/km


In [4]:
def generate_delivery_demand(restaurants_gdf, n_deliveries, origin_lat, origin_lon, 
                              max_radius_km=10, avg_parcel_weight_kg=5):
    """
    Generate delivery demand by sampling restaurants within radius of dispatch center
    
    Parameters:
    - n_deliveries: number of deliveries to generate
    - max_radius_km: only sample restaurants within this radius
    - avg_parcel_weight_kg: average weight per parcel (for payload constraints)
    
    Returns: GeoDataFrame with delivery locations and parcel info
    """
    
    # Filter restaurants within max_radius using Haversine
    def haversine(lat1, lon1, lat2, lon2):
        from math import radians, sin, cos, sqrt, atan2
        R = 6371  # Earth radius in km
        
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        return R * c
    
    # Calculate distances
    restaurants_gdf['distance_km'] = restaurants_gdf.apply(
        lambda row: haversine(origin_lat, origin_lon, row.geometry.y, row.geometry.x),
        axis=1
    )
    
    # Filter within radius
    nearby = restaurants_gdf[restaurants_gdf['distance_km'] <= max_radius_km].copy()
    
    if len(nearby) < n_deliveries:
        print(f"Warning: Only {len(nearby)} restaurants within {max_radius_km}km, requested {n_deliveries}")
        n_deliveries = len(nearby)
    
    # Sample deliveries
    deliveries = nearby.sample(n=n_deliveries, replace=False).copy()
    
    # Add parcel info (with some variation)
    deliveries['parcel_weight_kg'] = np.random.normal(avg_parcel_weight_kg, avg_parcel_weight_kg*0.3, n_deliveries)
    deliveries['parcel_weight_kg'] = deliveries['parcel_weight_kg'].clip(lower=1)  # minimum 1kg
    
    deliveries = deliveries.reset_index(drop=True)
    deliveries['delivery_id'] = range(len(deliveries))
    
    return deliveries[['delivery_id', 'dba', 'geometry', 'distance_km', 'parcel_weight_kg']]


# Test it - generate 100 deliveries
test_deliveries = generate_delivery_demand(
    restaurants_gdf, 
    n_deliveries=100,
    origin_lat=origin_lat,
    origin_lon=origin_lon,
    max_radius_km=8,
    avg_parcel_weight_kg=5
)

print(f"Generated {len(test_deliveries)} deliveries")
print(f"Distance range: {test_deliveries['distance_km'].min():.2f} - {test_deliveries['distance_km'].max():.2f} km")
print(f"Weight range: {test_deliveries['parcel_weight_kg'].min():.2f} - {test_deliveries['parcel_weight_kg'].max():.2f} kg")
print(f"\nFirst 5 deliveries:")
print(test_deliveries.head())

Generated 100 deliveries
Distance range: 0.16 - 7.94 km
Weight range: 1.21 - 8.60 kg

First 5 deliveries:
   delivery_id                    dba                    geometry  \
0            0           POLLOS MARIO  POINT (-73.90256 40.74462)   
1            1            DUTCH FREDS  POINT (-73.98802 40.76064)   
2            2  GRAND SICHUAN EASTERN   POINT (-73.9661 40.75821)   
3            3       INSOMNIA COOKIES  POINT (-74.00869 40.71396)   
4            4         YUE LAI BAKERY  POINT (-73.99165 40.71391)   

   distance_km  parcel_weight_kg  
0     6.869493          4.258392  
1     0.771730          4.641475  
2     1.336099          5.499817  
3     6.066694          6.118022  
4     5.655248          3.617057  


In [5]:
class RouteChromosome:
    """
    Represents one possible solution (route sequence) for deliveries
    """
    
    def __init__(self, delivery_ids):
        """
        delivery_ids: list of delivery IDs in visit order
        """
        self.genes = delivery_ids.copy()  # The sequence
        self.fitness = None  # Will be calculated later
        self.routes = []  # Will be split into trips
        self.total_distance = 0
        self.total_cost = 0
        self.total_emissions = 0
        self.feasible = True
    
    def __repr__(self):
        fitness_str = f"{self.fitness:.2f}" if self.fitness is not None else "None"
        return f"Chromosome(fitness={fitness_str}, genes={self.genes[:5]}...)"
    
    @staticmethod
    def random_chromosome(n_deliveries):
        """Create a random route sequence"""
        delivery_ids = list(range(n_deliveries))
        random.shuffle(delivery_ids)
        return RouteChromosome(delivery_ids)


# Test it
test_chromosome = RouteChromosome.random_chromosome(10)
print(f"Random chromosome: {test_chromosome.genes}")
print(f"Chromosome object: {test_chromosome}")

Random chromosome: [0, 5, 9, 1, 7, 4, 6, 2, 3, 8]
Chromosome object: Chromosome(fitness=None, genes=[0, 5, 9, 1, 7]...)


In [6]:
# Reload updated parameters
perf = pd.read_excel('../Cleaned Data/Mode_performance_parameters.xlsx')
cost = pd.read_excel('../Cleaned Data/Mode_cost_parameters.xlsx')

# Rebuild mode params
mode_params = ModeParameters(perf, cost)

# Add mode_name and parcels_per_stop to each mode
parcels_per_stop = {
    'Truck': 5,
    'E-Van': 2,
    'Cargo Bike': 1
}

for mode_name, params in mode_params.params.items():
    params['mode_name'] = mode_name
    params['parcels_per_stop'] = parcels_per_stop[mode_name]

# Verify
print("=== VERIFIED MODE PARAMETERS ===\n")
for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
    p = mode_params.get(mode_name)
    calculated = int(p['stops_per_trip']) * p['parcels_per_stop'] * int(p['trips_per_day'])
    stated = p['max_parcels_per_day']
    match = "✓" if abs(calculated - stated) <= 2 else "✗ MISMATCH"
    
    print(f"{mode_name}:")
    print(f"  Stops/trip: {int(p['stops_per_trip'])}")
    print(f"  Parcels/stop: {p['parcels_per_stop']}")
    print(f"  Trips/day: {int(p['trips_per_day'])}")
    print(f"  Calculated max parcels/day: {calculated}")
    print(f"  Stated max parcels/day: {stated} {match}")
    print(f"  Max payload: {p['max_payload_kg']} kg")
    print(f"  Max service radius: {p['max_service_radius_km']} km")
    print(f"  Cost/km: ${p['cost_per_km']}")
    print(f"  Emissions: {p['emission_factor_co2_per_km']} kg CO2/km")
    print()

=== VERIFIED MODE PARAMETERS ===

Cargo Bike:
  Stops/trip: 13
  Parcels/stop: 1
  Trips/day: 3
  Calculated max parcels/day: 39
  Stated max parcels/day: 39 ✓
  Max payload: 300 kg
  Max service radius: 6 km
  Cost/km: $0.25
  Emissions: 0.0155 kg CO2/km

E-Van:
  Stops/trip: 40
  Parcels/stop: 2
  Trips/day: 2
  Calculated max parcels/day: 160
  Stated max parcels/day: 160 ✓
  Max payload: 1000 kg
  Max service radius: 30 km
  Cost/km: $0.7
  Emissions: 0.1575 kg CO2/km

Truck:
  Stops/trip: 25
  Parcels/stop: 5
  Trips/day: 1
  Calculated max parcels/day: 125
  Stated max parcels/day: 125 ✓
  Max payload: 1200 kg
  Max service radius: 40 km
  Cost/km: $1.2
  Emissions: 0.3235 kg CO2/km



In [7]:
def split_into_trips_v3(chromosome_genes, deliveries_df, mode_params):
    """
    Final trip splitter incorporating parcels_per_stop correctly.
    
    Logic:
    - Each gene = one STOP (which may contain multiple parcels)
    - Cargo Bike: 1 parcel/stop, 13 stops/trip, 3 trips/day
    - E-Van:      2 parcels/stop, 40 stops/trip, 2 trips/day  
    - Truck:      5 parcels/stop, 25 stops/trip, 1 trip/day
    
    Constraints enforced per trip:
    1. max stops per trip
    2. max payload (kg) per trip
    3. max trips per day
    4. max parcels per day (hard ceiling)
    """
    
    max_payload_kg      = mode_params['max_payload_kg']
    stops_per_trip      = int(mode_params['stops_per_trip'])
    trips_per_day       = int(mode_params['trips_per_day'])
    max_parcels_per_day = mode_params['max_parcels_per_day']
    parcels_per_stop    = mode_params['parcels_per_stop']
    
    trips = []           # list of completed trips
    current_trip = []    # stops in current trip
    current_weight = 0   # kg loaded in current trip
    current_stops = 0    # stops made in current trip
    total_parcels = 0    # parcels delivered today across all trips
    unserved = []        # stops that couldn't be served
    
    for stop_id in chromosome_genes:
        
        # Hard ceiling: max parcels per day reached
        if total_parcels >= max_parcels_per_day:
            unserved.append(stop_id)
            continue
        
        # Get weight for this stop
        stop_weight = deliveries_df.loc[
            deliveries_df['delivery_id'] == stop_id, 'parcel_weight_kg'
        ].values[0] * parcels_per_stop
        
        # Check if adding this stop would exceed trip constraints
        weight_exceeded = (current_weight + stop_weight) > max_payload_kg
        stops_exceeded  = current_stops >= stops_per_trip
        
        if weight_exceeded or stops_exceeded:
            
            # Save completed trip
            if current_trip:
                trips.append(current_trip.copy())
                total_parcels += len(current_trip) * parcels_per_stop
            
            # Check if we've used all trips for today
            if len(trips) >= trips_per_day:
                unserved.append(stop_id)
                continue
            
            # Start new trip with this stop
            current_trip   = [stop_id]
            current_weight = stop_weight
            current_stops  = 1
        
        else:
            # Add stop to current trip
            current_trip.append(stop_id)
            current_weight += stop_weight
            current_stops  += 1
    
    # Save last trip if within daily limit
    if current_trip:
        if len(trips) < trips_per_day:
            trips.append(current_trip.copy())
            total_parcels += len(current_trip) * parcels_per_stop
        else:
            unserved.extend(current_trip)
    
    return trips, unserved, total_parcels


# ── Test across three demand levels ──────────────────────────────────────────

demand_levels = [50, 100, 200]

for n_deliveries in demand_levels:
    
    # Generate demand for this level
    deliveries = generate_delivery_demand(
        restaurants_gdf,
        n_deliveries   = n_deliveries,
        origin_lat     = origin_lat,
        origin_lon     = origin_lon,
        max_radius_km  = 8,
        avg_parcel_weight_kg = 5
    )
    
    genes = list(range(n_deliveries))
    random.shuffle(genes)
    
    print(f"\n{'='*55}")
    print(f"DEMAND LEVEL: {n_deliveries} stops")
    print(f"{'='*55}")
    
    for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
        p = mode_params.get(mode_name)
        
        trips, unserved, total_parcels = split_into_trips_v3(
            genes, deliveries, p
        )
        
        total_stops_served = sum(len(t) for t in trips)
        service_rate = (total_stops_served / n_deliveries) * 100
        
        print(f"\n  {mode_name}:")
        print(f"    Trips used:        {len(trips)}/{p['trips_per_day']}")
        print(f"    Stops served:      {total_stops_served}/{n_deliveries}")
        print(f"    Parcels delivered: {total_parcels}")
        print(f"    Unserved stops:    {len(unserved)}")
        print(f"    Service rate:      {service_rate:.1f}%")


DEMAND LEVEL: 50 stops

  Cargo Bike:
    Trips used:        3/3
    Stops served:      39/50
    Parcels delivered: 39
    Unserved stops:    24
    Service rate:      78.0%

  E-Van:
    Trips used:        2/2
    Stops served:      50/50
    Parcels delivered: 100
    Unserved stops:    0
    Service rate:      100.0%

  Truck:
    Trips used:        1/1
    Stops served:      25/50
    Parcels delivered: 125
    Unserved stops:    50
    Service rate:      50.0%

DEMAND LEVEL: 100 stops

  Cargo Bike:
    Trips used:        3/3
    Stops served:      39/100
    Parcels delivered: 39
    Unserved stops:    74
    Service rate:      39.0%

  E-Van:
    Trips used:        2/2
    Stops served:      80/100
    Parcels delivered: 160
    Unserved stops:    60
    Service rate:      80.0%

  Truck:
    Trips used:        1/1
    Stops served:      25/100
    Parcels delivered: 125
    Unserved stops:    100
    Service rate:      25.0%

DEMAND LEVEL: 200 stops

  Cargo Bike:
    Trips u

In [8]:
def split_into_trips_v4(chromosome_genes, deliveries_df, mode_params):
    """
    Clean rewrite with explicit state tracking.
    Invariant: every stop_id ends up in exactly ONE of:
        - a trip (served)
        - out_of_range
        - unserved
    """
    
    max_payload_kg      = mode_params['max_payload_kg']
    stops_per_trip      = int(mode_params['stops_per_trip'])
    trips_per_day       = int(mode_params['trips_per_day'])
    max_parcels_per_day = mode_params['max_parcels_per_day']
    parcels_per_stop    = mode_params['parcels_per_stop']
    max_radius_km       = mode_params['max_service_radius_km']
    
    trips        = []   # completed trips
    current_trip = []   # stops in current open trip
    curr_weight  = 0.0
    curr_stops   = 0
    total_parcs  = 0
    
    out_of_range = []
    unserved     = []
    
    def close_current_trip():
        """Save current trip to trips list. Return True if saved."""
        nonlocal current_trip, curr_weight, curr_stops
        if current_trip:
            trips.append(current_trip.copy())
            current_trip = []
            curr_weight  = 0.0
            curr_stops   = 0
            return True
        return False
    
    for stop_id in chromosome_genes:
        
        # ── 1. Distance check ──────────────────────────────────────────
        stop_dist = deliveries_df.loc[
            deliveries_df['delivery_id'] == stop_id, 'distance_km'
        ].values[0]
        
        if stop_dist > max_radius_km:
            out_of_range.append(stop_id)
            continue   # skip — not served, not unserved by capacity
        
        # ── 2. Daily parcel ceiling ────────────────────────────────────
        if total_parcs >= max_parcels_per_day:
            unserved.append(stop_id)
            continue
        
        # ── 3. Get stop weight ─────────────────────────────────────────
        stop_wt = deliveries_df.loc[
            deliveries_df['delivery_id'] == stop_id, 'parcel_weight_kg'
        ].values[0] * parcels_per_stop
        
        # ── 4. Check if this stop fits in current trip ─────────────────
        fits = (
            (curr_weight + stop_wt) <= max_payload_kg and
            curr_stops < stops_per_trip
        )
        
        if fits:
            # Add to current trip
            current_trip.append(stop_id)
            curr_weight += stop_wt
            curr_stops  += 1
        
        else:
            # Current trip is full — close it
            if close_current_trip():
                total_parcs += len(trips[-1]) * parcels_per_stop
            
            # Can we start a new trip today?
            if len(trips) < trips_per_day:
                # Start new trip with this stop
                current_trip = [stop_id]
                curr_weight  = stop_wt
                curr_stops   = 1
            else:
                # No more trips available today
                unserved.append(stop_id)
    
    # ── 5. Close final open trip ───────────────────────────────────────
    if current_trip:
        if len(trips) < trips_per_day:
            if close_current_trip():
                total_parcs += len(trips[-1]) * parcels_per_stop
        else:
            unserved.extend(current_trip)
            current_trip = []
    
    return trips, unserved, out_of_range, total_parcs


# ── Validation test ────────────────────────────────────────────────────────

demand_levels = [50, 100, 200]

for n_deliveries in demand_levels:
    
    deliveries = generate_delivery_demand(
        restaurants_gdf,
        n_deliveries         = n_deliveries,
        origin_lat           = origin_lat,
        origin_lon           = origin_lon,
        max_radius_km        = 15,
        avg_parcel_weight_kg = 5
    )
    
    genes = list(range(n_deliveries))
    random.shuffle(genes)
    
    print(f"\n{'='*55}")
    print(f"DEMAND LEVEL: {n_deliveries} stops (pool: 15km)")
    print(f"{'='*55}")
    
    for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
        p = mode_params.get(mode_name)
        
        trips, unserved, oor, total_parcels = split_into_trips_v4(
            genes, deliveries, p
        )
        
        served       = sum(len(t) for t in trips)
        service_rate = (served / n_deliveries) * 100
        total_acc    = served + len(unserved) + len(oor)
        check        = "✓" if total_acc == n_deliveries else f"✗ got {total_acc}"
        
        print(f"\n  {mode_name} (radius: {p['max_service_radius_km']}km):")
        print(f"    Trips used:          {len(trips)}/{p['trips_per_day']}")
        print(f"    Stops served:        {served}/{n_deliveries}")
        print(f"    Parcels delivered:   {total_parcels}")
        print(f"    Out of range:        {len(oor)}")
        print(f"    Unserved (capacity): {len(unserved)}")
        print(f"    Service rate:        {service_rate:.1f}%")
        print(f"    Counts check:        {check}")


DEMAND LEVEL: 50 stops (pool: 15km)

  Cargo Bike (radius: 6km):
    Trips used:          3/3
    Stops served:        27/50
    Parcels delivered:   27
    Out of range:        23
    Unserved (capacity): 0
    Service rate:        54.0%
    Counts check:        ✓

  E-Van (radius: 30km):
    Trips used:          2/2
    Stops served:        50/50
    Parcels delivered:   100
    Out of range:        0
    Unserved (capacity): 0
    Service rate:        100.0%
    Counts check:        ✓

  Truck (radius: 40km):
    Trips used:          1/1
    Stops served:        25/50
    Parcels delivered:   125
    Out of range:        0
    Unserved (capacity): 25
    Service rate:        50.0%
    Counts check:        ✓

DEMAND LEVEL: 100 stops (pool: 15km)

  Cargo Bike (radius: 6km):
    Trips used:          3/3
    Stops served:        39/100
    Parcels delivered:   39
    Out of range:        51
    Unserved (capacity): 10
    Service rate:        39.0%
    Counts check:        ✓

  E-Van 

In [9]:
def calculate_fitness(chromosome_genes, deliveries_df, mode_params, G, origin_node):
    """
    Evaluate how good a chromosome (route sequence) is.
    
    Steps:
    1. Split genes into trips
    2. For each trip, calculate actual road distance via OSMnx
    3. Calculate total cost and emissions
    4. Return fitness score (lower = better)
    
    Returns dict with full breakdown for analysis
    """
    
    mode_name    = mode_params['mode_name']
    cost_per_km  = mode_params['cost_per_km']
    emission_fac = mode_params['emission_factor_co2_per_km']
    speed_kmh    = mode_params['speed_kmh']
    
    # Step 1: Split into trips
    trips, unserved, out_of_range, total_parcels = split_into_trips_v4(
        chromosome_genes, deliveries_df, mode_params
    )
    
    total_distance_km  = 0.0
    total_cost         = 0.0
    total_emissions    = 0.0
    total_time_hrs     = 0.0
    trip_details       = []
    
    # Step 2: Calculate distance for each trip
    for trip_idx, trip in enumerate(trips):
        
        trip_distance = 0.0
        prev_node     = origin_node  # start from depot
        
        for stop_id in trip:
            
            # Get stop location
            stop_row  = deliveries_df[deliveries_df['delivery_id'] == stop_id].iloc[0]
            stop_node = ox.nearest_nodes(G, stop_row.geometry.x, stop_row.geometry.y)
            
            # Get road distance to this stop
            try:
                path_length = nx.shortest_path_length(
                    G, prev_node, stop_node, weight='length'
                )
                trip_distance += path_length / 1000  # convert m to km
            except nx.NetworkXNoPath:
                trip_distance += stop_row['distance_km'] * 1.3  # fallback
            
            prev_node = stop_node
        
        # Return to depot
        try:
            return_length = nx.shortest_path_length(
                G, prev_node, origin_node, weight='length'
            )
            trip_distance += return_length / 1000
        except nx.NetworkXNoPath:
            pass
        
        # Calculate metrics for this trip
        trip_cost      = trip_distance * cost_per_km
        trip_emissions = trip_distance * emission_fac
        trip_time      = trip_distance / speed_kmh
        
        total_distance_km += trip_distance
        total_cost        += trip_cost
        total_emissions   += trip_emissions
        total_time_hrs    += trip_time
        
        trip_details.append({
            'trip_idx'     : trip_idx + 1,
            'stops'        : len(trip),
            'distance_km'  : round(trip_distance, 3),
            'cost'         : round(trip_cost, 3),
            'emissions_kg' : round(trip_emissions, 3),
            'time_hrs'     : round(trip_time, 3)
        })
    
    # Step 3: Calculate fitness score
    # Penalty for unserved stops (we want to serve as many as possible)
    n_total    = len(chromosome_genes)
    n_served   = sum(len(t) for t in trips)
    n_unserved = len(unserved)
    
    unserved_penalty = n_unserved * 10  # penalty per unserved stop
    
    # Fitness = total cost + emissions cost + unserved penalty
    # Lower is better
    emissions_weight = 2.0  # how much we penalize emissions vs cost
    fitness = total_cost + (total_emissions * emissions_weight) + unserved_penalty
    
    return {
        'fitness'          : round(fitness, 3),
        'total_distance_km': round(total_distance_km, 3),
        'total_cost'       : round(total_cost, 3),
        'total_emissions'  : round(total_emissions, 3),
        'total_time_hrs'   : round(total_time_hrs, 3),
        'trips_used'       : len(trips),
        'stops_served'     : n_served,
        'stops_unserved'   : n_unserved,
        'out_of_range'     : len(out_of_range),
        'service_rate'     : round(n_served / n_total * 100, 1),
        'total_parcels'    : total_parcels,
        'trip_details'     : trip_details
    }


# ── Test fitness on one chromosome ────────────────────────────────────────

print("Testing fitness function (this will take ~30 seconds)...")
print("OSMnx is computing real road distances for each stop\n")

# Generate 50 deliveries
test_deliveries_50 = generate_delivery_demand(
    restaurants_gdf,
    n_deliveries         = 50,
    origin_lat           = origin_lat,
    origin_lon           = origin_lon,
    max_radius_km        = 15,
    avg_parcel_weight_kg = 5
)

test_genes_50 = list(range(50))
random.shuffle(test_genes_50)

for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
    p = mode_params.get(mode_name)
    
    result = calculate_fitness(
        test_genes_50, test_deliveries_50, p, G, origin_node
    )
    
    print(f"{mode_name}:")
    print(f"  Fitness score:     {result['fitness']}")
    print(f"  Total distance:    {result['total_distance_km']} km")
    print(f"  Total cost:        ${result['total_cost']}")
    print(f"  Total emissions:   {result['total_emissions']} kg CO2")
    print(f"  Service rate:      {result['service_rate']}%")
    print(f"  Time to complete:  {result['total_time_hrs']} hrs")
    print()

Testing fitness function (this will take ~30 seconds)...
OSMnx is computing real road distances for each stop

Cargo Bike:
  Fitness score:     29.521
  Total distance:    105.058 km
  Total cost:        $26.265
  Total emissions:   1.628 kg CO2
  Service rate:      42.0%
  Time to complete:  4.202 hrs

E-Van:
  Fitness score:     423.852
  Total distance:    417.588 km
  Total cost:        $292.312
  Total emissions:   65.77 kg CO2
  Service rate:      100.0%
  Time to complete:  8.352 hrs

Truck:
  Fitness score:     659.983
  Total distance:    221.972 km
  Total cost:        $266.367
  Total emissions:   71.808 kg CO2
  Service rate:      50.0%
  Time to complete:  5.549 hrs



In [10]:
def precompute_nodes(deliveries_df, G):
    """
    Pre-compute OSMnx nearest nodes for all delivery locations.
    This runs ONCE and saves enormous computation time during GA.
    """
    print(f"Pre-computing nearest nodes for {len(deliveries_df)} locations...")
    
    nodes = {}
    for idx, row in deliveries_df.iterrows():
        stop_id = row['delivery_id']
        nodes[stop_id] = ox.nearest_nodes(G, row.geometry.x, row.geometry.y)
    
    print(f"Done. {len(nodes)} nodes cached.")
    return nodes


def calculate_fitness_fast(chromosome_genes, deliveries_df, mode_params, 
                           G, origin_node, precomputed_nodes):
    """
    Fast version using pre-computed nodes.
    Same logic as calculate_fitness but no repeated node lookups.
    """
    
    mode_name    = mode_params['mode_name']
    cost_per_km  = mode_params['cost_per_km']
    emission_fac = mode_params['emission_factor_co2_per_km']
    speed_kmh    = mode_params['speed_kmh']
    
    trips, unserved, out_of_range, total_parcels = split_into_trips_v4(
        chromosome_genes, deliveries_df, mode_params
    )
    
    total_distance_km = 0.0
    total_cost        = 0.0
    total_emissions   = 0.0
    total_time_hrs    = 0.0
    trip_details      = []
    
    for trip_idx, trip in enumerate(trips):
        
        trip_distance = 0.0
        prev_node     = origin_node
        
        for stop_id in trip:
            stop_node = precomputed_nodes[stop_id]  # instant lookup
            
            try:
                path_length    = nx.shortest_path_length(
                    G, prev_node, stop_node, weight='length'
                )
                trip_distance += path_length / 1000
            except nx.NetworkXNoPath:
                stop_row       = deliveries_df[deliveries_df['delivery_id'] == stop_id].iloc[0]
                trip_distance += stop_row['distance_km'] * 1.3
            
            prev_node = stop_node
        
        # Return to depot
        try:
            return_length  = nx.shortest_path_length(
                G, prev_node, origin_node, weight='length'
            )
            trip_distance += return_length / 1000
        except nx.NetworkXNoPath:
            pass
        
        trip_cost      = trip_distance * cost_per_km
        trip_emissions = trip_distance * emission_fac
        trip_time      = trip_distance / speed_kmh
        
        total_distance_km += trip_distance
        total_cost        += trip_cost
        total_emissions   += trip_emissions
        total_time_hrs    += trip_time
        
        trip_details.append({
            'trip_idx'    : trip_idx + 1,
            'stops'       : len(trip),
            'distance_km' : round(trip_distance, 3),
            'cost'        : round(trip_cost, 3),
            'emissions_kg': round(trip_emissions, 3),
            'time_hrs'    : round(trip_time, 3)
        })
    
    n_total    = len(chromosome_genes)
    n_served   = sum(len(t) for t in trips)

    n_unserved = len(unserved) + len(out_of_range)
    unserved_penalty = n_unserved * 10
    
    emissions_weight  = 2.0
    fitness = total_cost + (total_emissions * emissions_weight) + unserved_penalty
    
    return {
        'fitness'          : round(fitness, 3),
        'total_distance_km': round(total_distance_km, 3),
        'total_cost'       : round(total_cost, 3),
        'total_emissions'  : round(total_emissions, 3),
        'total_time_hrs'   : round(total_time_hrs, 3),
        'trips_used'       : len(trips),
        'stops_served'     : n_served,
        'stops_unserved'   : n_unserved,
        'out_of_range'     : len(out_of_range),
        'service_rate'     : round(n_served / n_total * 100, 1),
        'total_parcels'    : total_parcels,
        'trip_details'     : trip_details
    }


# Pre-compute nodes for our test deliveries
import time

start = time.time()
precomputed_nodes_50 = precompute_nodes(test_deliveries_50, G)
elapsed = time.time() - start
print(f"Node pre-computation took {elapsed:.1f} seconds")

# Test speed improvement
print("\nTesting fast fitness function...")
start = time.time()

for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
    p = mode_params.get(mode_name)
    result = calculate_fitness_fast(
        test_genes_50, test_deliveries_50, p, 
        G, origin_node, precomputed_nodes_50
    )
    print(f"{mode_name}: fitness={result['fitness']}, "
          f"distance={result['total_distance_km']}km, "
          f"service={result['service_rate']}%")

elapsed = time.time() - start
print(f"\nFast version took {elapsed:.1f} seconds")
print("(Original version took ~30-60 seconds)")

Pre-computing nearest nodes for 50 locations...
Done. 50 nodes cached.
Node pre-computation took 0.4 seconds

Testing fast fitness function...
Cargo Bike: fitness=319.521, distance=105.058km, service=42.0%
E-Van: fitness=423.852, distance=417.588km, service=100.0%
Truck: fitness=659.983, distance=221.972km, service=50.0%

Fast version took 0.5 seconds
(Original version took ~30-60 seconds)


In [11]:
import random
import numpy as np
from copy import deepcopy

# ── GA Parameters ─────────────────────────────────────────────────────────

GA_PARAMS = {
    'population_size' : 60,    # chromosomes per generation
    'n_generations'   : 100,   # how many generations to evolve
    'elite_size'      : 5,     # top N survivors kept unchanged
    'crossover_rate'  : 0.85,  # probability of crossover vs cloning
    'mutation_rate'   : 0.15,  # probability of mutating a chromosome
    'tournament_size' : 4,     # candidates per tournament selection
}


# ── Selection: Tournament ──────────────────────────────────────────────────

def tournament_select(population, fitnesses, k=4):
    """
    Pick k random chromosomes, return the one with lowest fitness.
    Lower fitness = better solution.
    """
    candidates = random.sample(range(len(population)), k)
    best       = min(candidates, key=lambda i: fitnesses[i])
    return population[best][:]   # return a copy


# ── Crossover: Order Crossover (OX1) ──────────────────────────────────────

def order_crossover(parent_a, parent_b):
    """
    OX1 crossover — preserves relative order of genes.
    Standard operator for routing/TSP problems.
    
    Example:
      Parent A: [0,1,2,3,4,5,6,7]
      Parent B: [3,5,1,7,0,4,2,6]
      Slice:    [  _ _ 2 3 4 _ _]  (positions 2-4 from A)
      Fill:     remaining in order from B → [5,1,7,0,4,2,6] minus [2,3,4]
      Child:    [5,1, 2,3,4, 7,0,6]
    
    This is critical for routing: it respects that each stop appears
    exactly once (no duplicates, no missing stops).
    """
    n     = len(parent_a)
    start = random.randint(0, n - 2)
    end   = random.randint(start + 1, n)
    
    # Copy slice from parent A
    child          = [None] * n
    child[start:end] = parent_a[start:end]
    
    # Fill remaining positions from parent B in order
    remaining = [g for g in parent_b if g not in child[start:end]]
    pos       = 0
    for i in range(n):
        if child[i] is None:
            child[i] = remaining[pos]
            pos      += 1
    
    return child


# ── Mutation: Swap Mutation ────────────────────────────────────────────────

def swap_mutate(chromosome):
    """
    Randomly swap two genes.
    Simple but effective for routing problems.
    """
    n   = len(chromosome)
    i, j = random.sample(range(n), 2)
    chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
    return chromosome


def reverse_segment_mutate(chromosome):
    """
    Reverse a random segment (2-opt style).
    Often more effective than swap for geographic routes.
    """
    n     = len(chromosome)
    start = random.randint(0, n - 2)
    end   = random.randint(start + 1, n)
    chromosome[start:end] = chromosome[start:end][::-1]
    return chromosome


def mutate(chromosome, mutation_rate):
    """Apply mutation with given probability."""
    if random.random() < mutation_rate:
        # 50/50 between swap and reverse-segment
        if random.random() < 0.5:
            return swap_mutate(chromosome)
        else:
            return reverse_segment_mutate(chromosome)
    return chromosome


# ── Main GA Loop ───────────────────────────────────────────────────────────

def run_ga(deliveries_df, mode_params, G, origin_node,
           precomputed_nodes, ga_params=GA_PARAMS, verbose=True):
    """
    Full GA for one mode on one demand set.
    
    Returns:
        best_chromosome : gene sequence with lowest fitness found
        best_result     : full fitness breakdown of best solution
        history         : fitness per generation (for convergence plot)
    """
    
    n_stops  = len(deliveries_df)
    genes    = list(range(n_stops))
    pop_size = ga_params['population_size']
    n_gen    = ga_params['n_generations']
    elite_n  = ga_params['elite_size']
    c_rate   = ga_params['crossover_rate']
    m_rate   = ga_params['mutation_rate']
    t_size   = ga_params['tournament_size']
    
    mode_name = mode_params['mode_name']
    
    # ── Initialize population ──────────────────────────────────────────────
    population = []
    for _ in range(pop_size):
        shuffled = genes[:]
        random.shuffle(shuffled)
        population.append(shuffled)
    
    # ── Evaluate initial population ────────────────────────────────────────
    fitnesses = [
        calculate_fitness_fast(
            chrom, deliveries_df, mode_params,
            G, origin_node, precomputed_nodes
        )['fitness']
        for chrom in population
    ]
    
    best_fitness   = min(fitnesses)
    best_chrom_idx = fitnesses.index(best_fitness)
    best_chrom     = population[best_chrom_idx][:]
    history        = [best_fitness]
    
    if verbose:
        print(f"\n{mode_name} — Starting GA")
        print(f"  Population: {pop_size} | Generations: {n_gen}")
        print(f"  Initial best fitness: {best_fitness:.3f}")
        print(f"  {'Gen':>5}  {'Best':>10}  {'Avg':>10}  {'Improvement':>12}")
        print(f"  {'-'*45}")
    
    # ── Evolve ────────────────────────────────────────────────────────────
    for gen in range(n_gen):
        
        # Sort by fitness (ascending — lower is better)
        sorted_pairs = sorted(zip(fitnesses, population), key=lambda x: x[0])
        fitnesses    = [p[0] for p in sorted_pairs]
        population   = [p[1] for p in sorted_pairs]
        
        new_population = []
        
        # Elitism: keep top N unchanged
        new_population.extend([chrom[:] for chrom in population[:elite_n]])
        
        # Fill rest with crossover + mutation
        while len(new_population) < pop_size:
            
            parent_a = tournament_select(population, fitnesses, t_size)
            
            if random.random() < c_rate:
                parent_b = tournament_select(population, fitnesses, t_size)
                child    = order_crossover(parent_a, parent_b)
            else:
                child    = parent_a[:]
            
            child = mutate(child, m_rate)
            new_population.append(child)
        
        # Evaluate new population
        population = new_population
        fitnesses  = [
            calculate_fitness_fast(
                chrom, deliveries_df, mode_params,
                G, origin_node, precomputed_nodes
            )['fitness']
            for chrom in population
        ]
        
        gen_best = min(fitnesses)
        gen_avg  = sum(fitnesses) / len(fitnesses)
        
        if gen_best < best_fitness:
            best_fitness   = gen_best
            best_chrom_idx = fitnesses.index(gen_best)
            best_chrom     = population[best_chrom_idx][:]
        
        history.append(best_fitness)
        
        # Print every 10 generations
        if verbose and (gen + 1) % 10 == 0:
            improvement = history[0] - best_fitness
            print(f"  {gen+1:>5}  {best_fitness:>10.3f}  "
                  f"{gen_avg:>10.3f}  {improvement:>12.3f}")
    
    # ── Final result ──────────────────────────────────────────────────────
    best_result = calculate_fitness_fast(
        best_chrom, deliveries_df, mode_params,
        G, origin_node, precomputed_nodes
    )
    
    if verbose:
        print(f"\n  ✓ Final best fitness: {best_fitness:.3f}")
        print(f"    Distance:    {best_result['total_distance_km']} km")
        print(f"    Cost:        ${best_result['total_cost']}")
        print(f"    Emissions:   {best_result['total_emissions']} kg CO2")
        print(f"    Service rate:{best_result['service_rate']}%")
        print(f"    Trips used:  {best_result['trips_used']}")
    
    return best_chrom, best_result, history


# ── Run GA for all three modes ─────────────────────────────────────────────

print("=" * 55)
print("RUNNING GA — 50 STOPS, 15km POOL")
print("=" * 55)

ga_results = {}

for mode_name in ['Cargo Bike', 'E-Van', 'Truck']:
    p = mode_params.get(mode_name)
    
    best_chrom, best_result, history = run_ga(
        test_deliveries_50, p, G,
        origin_node, precomputed_nodes_50
    )
    
    ga_results[mode_name] = {
        'chromosome' : best_chrom,
        'result'     : best_result,
        'history'    : history
    }

print("\n" + "=" * 55)
print("GA COMPLETE — SUMMARY")
print("=" * 55)
for mode_name, data in ga_results.items():
    r = data['result']
    print(f"\n{mode_name}:")
    print(f"  Fitness:      {r['fitness']}")
    print(f"  Distance:     {r['total_distance_km']} km")
    print(f"  Cost:         ${r['total_cost']}")
    print(f"  Emissions:    {r['total_emissions']} kg CO2")
    print(f"  Service rate: {r['service_rate']}%")
    print(f"  Trips used:   {r['trips_used']}")

RUNNING GA — 50 STOPS, 15km POOL

Cargo Bike — Starting GA
  Population: 60 | Generations: 100
  Initial best fitness: 315.184
    Gen        Best         Avg   Improvement
  ---------------------------------------------
     10     307.694     308.506         7.490
     20     304.380     305.682        10.804
     30     304.014     304.298        11.170
     40     303.885     304.113        11.299
     50     303.885     304.165        11.299
     60     303.739     304.004        11.445
     70     303.538     303.722        11.646
     80     303.538     303.817        11.646
     90     303.294     303.728        11.890
    100     303.294     303.479        11.890

  ✓ Final best fitness: 303.294
    Distance:    47.311 km
    Cost:        $11.828
    Emissions:   0.733 kg CO2
    Service rate:42.0%
    Trips used:  2

E-Van — Starting GA
  Population: 60 | Generations: 100
  Initial best fitness: 389.780
    Gen        Best         Avg   Improvement
  -------------------------

In [12]:
result = calculate_fitness_fast(
    test_genes_50, test_deliveries_50, 
    mode_params.params['Cargo Bike'], 
    G, origin_node, precomputed_nodes_50
)

print(f"Fitness:            {result['fitness']}")
print(f"Total cost:         {result['total_cost']}")
print(f"Emissions weighted: {round(result['total_emissions'] * 2.0, 3)}")
print(f"Stops served:       {result['stops_served']}")
print(f"Stops unserved:     {result['stops_unserved']}")
print(f"Out of range:       {result['out_of_range']}")
print(f"Unserved penalty:   {result['stops_unserved'] * 10}")

Fitness:            319.521
Total cost:         26.265
Emissions weighted: 3.256
Stops served:       21
Stops unserved:     29
Out of range:       29
Unserved penalty:   290
